In [13]:
import earthaccess
import os
import warnings
import csv
from osgeo import gdal
import numpy as np
import math
import rasterio as rio
import xarray as xr
import holoviews as hv
import hvplot.xarray
import netCDF4 as nc
#import cv2 as cv

import glob
import sys
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import Affine
from scipy import linalg
from lxml import etree
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation
from skimage.transform import hough_line, hough_line_peaks

from config import CONFIG, POWER_PLANTS, CROSS_SECTIONS, LOCS
sys.path.append('../EMIT-Data-Resources/python/modules/')
from emit_tools import emit_xarray, ortho_xr

from EMIT_NOX import run_retrieval

In [21]:
target_name = "Bridger"
target_type = 'plant'
# if target_type == 'plant':
#     target_data =  POWER_PLANTS[target_name]
# else:
#     target_data = LOCS[target_name]

samples = glob.glob(f"{CONFIG['data_folder']}/{target_name}/*_RAD_*.nc")
samples

['/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20230202T193810_2303313_012.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20230206T180234_2303712_009.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20240402T192441_2409313_021.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20240415T201449_2410613_021.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20240817T191022_2423013_020.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20241001T191815_2427513_015.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20241005T174220_2427912_018.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20241014T201312_2428813_020.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20250609T155951_2516010_004.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20250609T160003_2516010_005.nc',
 '/Volumes/T9/EMIT-NOX/data/EMIT/Bridger/EMIT_L1B_RAD_001_20250618T182535_2516912_020.nc',

In [19]:
fn = '/Volumes/T9/EMIT-NOX/data/EMIT/New_Madrid_Power_Plant/EMIT_L1B_RAD_001_20240815T191458_2422813_045.nc'# samples[0]
granule_name = fn.split('/')[-1]
granule_name

'EMIT_L1B_RAD_001_20240815T191458_2422813_045.nc'

In [44]:
POWER_PLANTS['Intermountain']

{'LON': -112.5811, 'LAT': 39.5035, 'NOx Mass (short tons)': 3691.782}

In [20]:
result_ds = run_retrieval(fn)
ortho_ds = ortho_xr(result_ds)

save_path = f"{CONFIG['results_folder']}/{target_name}"
os.makedirs(save_path, exist_ok=True)
ortho_ds.to_netcdf(f"{save_path}/{granule_name}") 

print(f"Saved product to {save_path}/{granule_name} !")

NOX cross section created! Computing DOAS0...
Found DOAS0! Computing plume mask...
Found plume mask! Computing DOAS...
DOAS Retrieval done! Converting to NetCDF...
Saved product to /Volumes/T9/EMIT-NOX/results/EMIT/New_Madrid_Power_Plant/EMIT_L1B_RAD_001_20240815T191458_2422813_045.nc !


In [ ]:
loc_name = 'Ninemile_Point'
fns = glob.glob(f'{CONFIG["results_folder"]}/{loc_name}/*.npy')

cols = 5
rows = (len(fns) + cols - 1)//cols
fig, axs = plt.subplots(rows,cols, figsize=(20,20))
axs = axs.flatten()



print(len(fns))
print('\n'.join(fns))

for i in range(min(len(fns), rows*cols)):
    dSCD = np.load(fns[i])
    gn_unmask = fns[i].split('/')[-1]
    radfn = f"{CONFIG['data_folder']}/{loc_name}/{gn_unmask[5:-4]}.nc"
    
    ds = emit_xarray(radfn)
    wl_val = float(ds["wavelengths"].isel(wavelengths=0))  # or a specific value
    dscd_da = xr.DataArray(
        dSCD.astype('float32')[..., None],  # -> (downtrack, crosstrack, 1)
        dims=("downtrack", "crosstrack", "wavelengths"),
        coords={
            "downtrack": ds["downtrack"],
            "crosstrack": ds["crosstrack"],
            "wavelengths": [wl_val],
        },
        name="dSCD",
        attrs={
            "long_name": "Differential Slant Column Density (single band)",
            "units": "molec cm^-2",
        },
    )
    ds = ds.assign(dSCD=dscd_da)
    ds = ortho_xr(ds)

    bounds = (np.min(ds['longitude'].values),
              np.min(ds['latitude'].values),
              np.max(ds['longitude'].values),
              np.max(ds['latitude'].values))

    axs[i].imshow(ds['radiance'][:,:,0], cmap='gray', extent=bounds)
    axs[i].imshow(ds['dSCD'][:,:,0]*1e19, cmap='RdBu_r', origin='upper', 
                  aspect='auto', vmin=-2e17, vmax=2e17, extent=bounds, alpha=0.7)
    axs[i].set_title(fns[i].split('/')[-1][-31:-4])

    
    
    # plt.imshow(plume_band*1e19, extent=bounds, cmap='RdBu_r')
    # cbar = plt.colorbar()
    # cbar.ax.set_ylabel('NOX dSCD (molec/cm2)', rotation=270, labelpad=15)
    # plt.title("NOX Retrieval - Morenci Mine")
    # plt.xlabel('longitude')
    # plt.ylabel('latitude')


fig.suptitle(loc_name, fontsize=24, y=0.92)
# plt.tight_layout()
plt.show()

10
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20230403T172921_2309312_002.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20231222T174800_2335612_015.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20240404T161230_2409511_009.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20240621T174325_2417312_047.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20240625T160755_2417711_031.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20240806T150710_2421910_004.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20240929T174236_2427312_012.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20250414T200042_2510413_035.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001_20250726T185148_2520712_016.npy
/Volumes/T9/EMIT-NOX/results/EMIT/Ninemile_Point/dSCD_EMIT_L1B_RAD_001